[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/02-census-block-groups.ipynb)

# Notebook 02 — Census Block Groups

Every spatial equity analysis begins with a fundamental question: *who lives within reach of this place?* To answer that, we need a way to divide geography into small, well-defined pieces that line up with published population data. In the United States, the Census Bureau provides exactly that through its hierarchy of **census geographies**.

This notebook introduces `get_census_blocks`, the SocialMapper function that retrieves **census block groups** — the smallest geographic unit for which the Census Bureau publishes detailed demographic data. By the end you will know how to:

1. Explain where block groups sit in the census geography hierarchy
2. Retrieve block groups that fall inside an isochrone polygon
3. Retrieve block groups around an arbitrary point and radius
4. Decode the 12-digit GEOID that uniquely identifies every block group
5. Visualize block group boundaries on a map colored by area
6. Understand how the number of block groups scales with search radius

---

## 1. The Census Geography Hierarchy

The Census Bureau organizes the country into a strict nesting of geographic units. Every level fits perfectly inside the level above it — no overlaps, no gaps:

```
Nation
 └── State                   (50 + DC + territories)
      └── County             (3,200+)
           └── Census Tract  (85,000+)   — designed for ~4,000 people
                └── Block Group (240,000+) — designed for 600–3,000 people
                     └── Block (11,000,000+) — smallest, but limited data
```

### Why block groups?

Census **blocks** (the very bottom of the hierarchy) are the most granular, but the bureau only publishes a handful of variables at that level (total population and basic race counts from the decennial census). For the rich demographic, economic, and housing data from the **American Community Survey (ACS)**, the block group is the finest resolution available.

That makes block groups the ideal unit for neighborhood-level analysis:

- **Small enough** to capture variation within a city (typically a few city blocks)
- **Data-rich enough** to include income, education, commute times, health insurance status, and hundreds of other variables
- **Published every year** as part of the ACS 5-year estimates, so the data stays reasonably current

SocialMapper uses block groups as its default unit of analysis throughout the library.

---

## 2. Setup

We import `create_isochrone` (to generate a travel-time polygon) and `get_census_blocks` (to find which block groups overlap that polygon). We also bring in `matplotlib` for visualizations later in the notebook.

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import create_isochrone, get_census_blocks

import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import contextily as cx

---

## 3. Retrieve Block Groups from an Isochrone

The most common workflow in SocialMapper is:

1. **Create an isochrone** — a polygon showing everywhere reachable within *N* minutes of driving, walking, or biking from a location.
2. **Find the block groups** that intersect that polygon.
3. **Pull demographics** for those block groups (covered in Notebook 03).

Let's start with step 1. We will generate a 15-minute drive-time isochrone centered on Austin, TX.

In [ ]:
iso = create_isochrone("Austin, TX", travel_time=15, travel_mode="drive")

print(f"Isochrone type:      {iso['type']}")
print(f"Geometry type:       {iso['geometry']['type']}")
print(f"Area covered:        {iso['properties']['area_sq_km']:.1f} sq km")
print(f"Travel time:         {iso['properties']['travel_time']} minutes")
print(f"Travel mode:         {iso['properties']['travel_mode']}")

Now we pass the isochrone directly to `get_census_blocks`. Under the hood, SocialMapper queries the Census Bureau's **TIGERweb** service — the authoritative source for census boundary geometries — and returns every block group whose boundary intersects the isochrone polygon.

In [ ]:
blocks = get_census_blocks(polygon=iso)

print(f"Block groups found: {len(blocks)}")
print(f"Return type:        {type(blocks).__name__} of {type(blocks[0]).__name__}s")

Each result is a plain Python dictionary. The function does not require `geopandas` or any heavy GIS dependencies — just standard dicts with GeoJSON geometry inside.

---

## 4. Inspect the Block Group Data Structure

Let's look at what each block group dictionary contains. Every entry has exactly seven keys:

In [ ]:
sample = blocks[0]

for key, value in sample.items():
    if key == "geometry":
        coords = value["coordinates"][0]
        print(f"  geometry:     <GeoJSON {value['type']} with {len(coords)} vertices>")
    else:
        print(f"  {key + ':':14s} {value}")

Here is what each field represents:

| Field | Type | Description |
|---|---|---|
| `geoid` | `str` | 12-digit unique identifier (see next section) |
| `state_fips` | `str` | 2-digit **Federal Information Processing Standards** code for the state |
| `county_fips` | `str` | 3-digit FIPS code for the county |
| `tract` | `str` | 6-digit census tract code |
| `block_group` | `str` | 1-digit block group number (1–9 within its tract) |
| `geometry` | `dict` | GeoJSON Polygon or MultiPolygon of the block group boundary |
| `area_sq_km` | `float` | Area of the block group in square kilometers |

---

## 5. GEOID Anatomy — Decoding the 12-Digit Identifier

Every block group in the United States has a unique **GEOID** (Geographic Identifier). It is a 12-character string that concatenates the FIPS codes from every level of the hierarchy. Think of it like a postal address, but for statistical geography.

**FIPS codes** (Federal Information Processing Standards) are numeric codes that the federal government assigns to every state, county, and smaller geography. They were created so that computers could unambiguously identify places — no confusion between "Washington" the state and "Washington" the county in 31 different states.

Here is the anatomy of a GEOID:

```
GEOID format:   S S C C C T T T T T T B
                ├─┤├──-┤├─────────┤├─┤
Position:       1-2 3-5   6-11     12
Level:         State County Tract  Block Group
```

Let's parse a real GEOID from our Austin results:

In [ ]:
geoid = sample["geoid"]

print(f"Full GEOID:   {geoid}")
print(f"              {''.join(['─'] * len(geoid))}")
print(f"State FIPS:   {geoid[:2]:4s}  (positions 1-2)   → {sample['state_fips']}")
print(f"County FIPS:  {geoid[2:5]:4s}  (positions 3-5)   → {sample['county_fips']}")
print(f"Tract:        {geoid[5:11]}  (positions 6-11)  → {sample['tract']}")
print(f"Block Group:  {geoid[11]:4s}  (position 12)     → {sample['block_group']}")
print()
print("Reconstruction check:")
reconstructed = sample["state_fips"] + sample["county_fips"] + sample["tract"] + sample["block_group"]
print(f"  {sample['state_fips']} + {sample['county_fips']} + {sample['tract']} + {sample['block_group']} = {reconstructed}")
print(f"  Matches GEOID? {reconstructed == geoid}")

The GEOID is important because it is the **join key** for all census data. When you download income, education, or population tables from the Census Bureau (or use SocialMapper's `get_census_data` in Notebook 03), you merge them to your block group geometries by GEOID. Every census table uses this exact same 12-digit identifier.

---

## 6. Visualize Block Group Boundaries

A table of GEOIDs is useful, but a map makes the spatial structure click. Let's plot every block group we retrieved, colored by its area in square kilometers. This reveals an important pattern: block groups in dense urban cores tend to be *physically small* (because they are drawn to contain a target population of 600–3,000 people, and dense areas pack more people per square kilometer).

In [ ]:
from shapely.geometry import shape

fig, ax = plt.subplots(figsize=(10, 10))

# Build a GeoDataFrame of block group polygons with area data
geometries = [shape(b["geometry"]) for b in blocks]
areas = [b["area_sq_km"] for b in blocks]
gdf_blocks = gpd.GeoDataFrame({"area_sq_km": areas}, geometry=geometries, crs="EPSG:4326")

# Plot block groups colored by area
gdf_blocks.plot(
    ax=ax, column="area_sq_km", cmap="YlOrRd", alpha=0.6,
    edgecolor="#555555", linewidth=0.5, legend=True,
    legend_kwds={"label": "Area (sq km)", "shrink": 0.7, "pad": 0.02},
)

# Overlay the isochrone boundary
iso_geom = shape(iso["geometry"])
gdf_iso = gpd.GeoDataFrame(geometry=[iso_geom], crs="EPSG:4326")
gdf_iso.boundary.plot(ax=ax, color="#d63384", linewidth=2.5, linestyle="--", label="15-min drive isochrone")

# Add road basemap
cx.add_basemap(ax, crs="EPSG:4326", source=cx.providers.CartoDB.Positron)

ax.set_title("Census Block Groups within 15-min Drive of Austin, TX", fontsize=14, fontweight="bold", color="#1a1a2e")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.show()

Notice how the smallest (lightest) block groups cluster near downtown Austin, while the larger (darker) ones spread out toward the suburbs. This is the Census Bureau's design at work — every block group targets roughly the same *population*, so higher-density areas produce smaller *areas*.

---

## 7. Point + Radius Lookup

Sometimes you do not have an isochrone — you just want to find block groups within a certain distance of a point. The `get_census_blocks` function supports this with the `location` and `radius_km` parameters.

Under the hood, SocialMapper constructs a circular polygon around your point and queries TIGERweb the same way. This is convenient when you want a quick geographic buffer without computing travel times.

In [ ]:
# University of Texas at Austin campus
ut_austin = (30.2849, -97.7341)

blocks_point = get_census_blocks(location=ut_austin, radius_km=3)
print(f"Block groups within 3 km of UT Austin: {len(blocks_point)}")

This is useful for quick exploratory analysis — for example, "how many block groups are near this hospital?" or "what does the census geography look like around this school?" For more rigorous accessibility analysis, use the isochrone approach from Section 3, since it accounts for the road network and real travel times rather than simple straight-line distance.

---

## 8. How Block Group Count Scales with Radius

How many block groups will you get as you increase the search radius? This matters for performance planning — larger areas mean more block groups, which means more API calls when you later fetch demographics.

Let's sweep through four radii and collect the counts.

In [ ]:
radii = [1, 3, 5, 10]
counts = []

for r in radii:
    result = get_census_blocks(location=ut_austin, radius_km=r)
    counts.append(len(result))
    print(f"  {r:>2} km radius  →  {len(result):>4} block groups")

print(f"\nGrowth factor (10 km vs 1 km): {counts[-1] / counts[0]:.1f}x")

Let's visualize this with a bar chart. Since the search area grows with the *square* of the radius (area = pi * r^2), we would expect the block group count to grow roughly quadratically — doubling the radius should give roughly 4x more block groups.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    [str(r) for r in radii],
    counts,
    color=["#4c9be8", "#3a7cc1", "#2a5f9e", "#1a3f6f"],
    edgecolor="#cccccc",
    linewidth=0.8,
    width=0.6,
)

# Add count labels on each bar
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.02,
        str(count),
        ha="center", va="bottom", fontsize=13, fontweight="bold", color="#1a1a2e",
    )

ax.set_xlabel("Search Radius (km)", fontsize=12)
ax.set_ylabel("Number of Block Groups", fontsize=12)
ax.set_title("Block Group Count by Search Radius (centered on UT Austin)", fontsize=13, fontweight="bold", color="#1a1a2e")
ax.set_ylim(0, max(counts) * 1.2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The near-quadratic growth confirms the spatial intuition. In practice, most analyses use a 15-minute drive isochrone (which typically covers 100–300 block groups in a metro area) — large enough for meaningful statistics, small enough to process quickly.

---

## 9. Distribution of Block Group Areas

Not all block groups are the same size. In fact, the *variation* in area tells you a lot about the density of the place you are studying. Let's look at the distribution of block group areas from our isochrone query using a histogram.

In [ ]:
areas = [b["area_sq_km"] for b in blocks]

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    areas,
    bins=25,
    color="#5ba37a",
    edgecolor="#ffffff",
    linewidth=0.8,
    alpha=0.85,
)

# Mark the median
median_area = sorted(areas)[len(areas) // 2]
ax.axvline(median_area, color="#d63384", linewidth=2, linestyle="--", label=f"Median: {median_area:.2f} sq km")

ax.set_xlabel("Block Group Area (sq km)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of Block Group Areas (Austin 15-min Drive)", fontsize=13, fontweight="bold", color="#1a1a2e")
ax.legend(fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Smallest block group: {min(areas):.3f} sq km")
print(f"Largest block group:  {max(areas):.3f} sq km")
print(f"Median area:          {median_area:.3f} sq km")
print(f"Ratio (max / min):    {max(areas) / min(areas):.0f}x")

The right-skewed distribution is typical of urban areas: most block groups are small and densely populated, but a few on the outskirts cover large tracts of lower-density land. This is worth keeping in mind when you later weight block groups by population rather than by area.

---

## 10. Summary Table with Pandas

Finally, let's compile the block group data into a tidy DataFrame for further analysis. This is typically the starting point before merging in demographic data from `get_census_data` (Notebook 03).

In [ ]:
df = pd.DataFrame([
    {
        "geoid": b["geoid"],
        "state_fips": b["state_fips"],
        "county_fips": b["county_fips"],
        "tract": b["tract"],
        "block_group": b["block_group"],
        "area_sq_km": round(b["area_sq_km"], 3),
    }
    for b in blocks
])

print(f"Total block groups:  {len(df)}")
print(f"Unique counties:     {df['county_fips'].nunique()}")
print(f"Unique tracts:       {df['tract'].nunique()}")
print(f"Total area:          {df['area_sq_km'].sum():.1f} sq km")
print(f"Mean area:           {df['area_sq_km'].mean():.3f} sq km")
print()
df.head(10)

---

## Summary

### Key Concepts

- **Census block groups** are the smallest geography with full ACS demographic data (income, education, commute, etc.).
- Each block group targets **600 to 3,000 people**, so physical area varies inversely with population density.
- The **12-digit GEOID** (`SSCCCTTTTTTB`) uniquely identifies every block group and serves as the join key for all census data tables.
- **FIPS codes** (Federal Information Processing Standards) are the numeric codes assigned to states and counties.
- Block group count grows roughly with the **square of the search radius**, since area = pi * r^2.

### API Reference

| Task | Code |
|---|---|
| Block groups inside a polygon | `get_census_blocks(polygon=iso)` |
| Block groups around a point | `get_census_blocks(location=(lat, lon), radius_km=5)` |
| Create an isochrone polygon | `create_isochrone("Austin, TX", travel_time=15, travel_mode="drive")` |

### Return Value

Both calling patterns return a `list[dict]` where each dict contains:

| Key | Example | Description |
|---|---|---|
| `geoid` | `"484530011041"` | 12-digit unique identifier |
| `state_fips` | `"48"` | State FIPS code |
| `county_fips` | `"453"` | County FIPS code |
| `tract` | `"001104"` | Census tract code |
| `block_group` | `"1"` | Block group number |
| `geometry` | `{"type": "Polygon", ...}` | GeoJSON boundary |
| `area_sq_km` | `0.547` | Area in square kilometers |

**Next notebook:** [03 — Census Demographics](03-census-demographics.ipynb)